# See-through · Kaggle T4×2 推理与公网 Web UI

图片 → 分层 PSD；仅推理，不包含训练。

- **网页**：Import Notebook 上传本文件，右侧确认 Accelerator = **GPU T4 x2**、Internet = **On**（元数据已预设）。
- **CLI**：本文件与 `kernel-metadata.json` 放在同目录；先把 metadata 的 `<username>` 改为自己的 Kaggle 用户名，再运行 `kaggle kernels push -p .`。
- **所有可调项只改 Cell 2 配置区**（下方第三个 cell，按 0 起算）。其余格子只读取配置。
- 默认 `RUN_BATCH=False`、`ENABLE_WEBUI=True`：Run All 在依赖安装后跳过 Cell 5–7，直接到 Cell 8 创建公网链接。仅批处理时在 Cell 2 设 `RUN_BATCH=True`、`ENABLE_WEBUI=False`；两者都开启会先等批处理、校验、打包完成后再启动 Web，不会同时占用 GPU。
- 默认 `REPO_URL` 指向包含本集成的 `BG-QWQ/see-through-kaggle`；也可使用其他包含这些文件的 fork 或 Kaggle Dataset 仓库副本。重跑 Cell 3 会将工作副本的 origin 对齐配置地址，再以 fast-forward-only 更新。


In [ ]:
# Cell 1 · 环境检查（无可调项）
import torch, os
print('torch', torch.__version__, 'CUDA', torch.version.cuda)
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), '请选择 GPU T4 x2 后重新启动 session'


In [ ]:
# Cell 2 · 唯一配置区：其余格子不需要编辑
# ============ 仓库来源与工作目录 ============
REPO_URL = 'https://github.com/BG-QWQ/see-through-kaggle.git'  # 包含 Kaggle 集成与逐请求隔离实现
REPO_BRANCH = 'main'  # 重跑 Cell 3 拉取新版；也支持固定 commit hash
REPO_DIR = '/kaggle/working/see-through'  # Dataset 可填 /kaggle/input/datasets/<owner>/<dataset>/<仓库目录>；以实际挂载为准
WORK_DIR = '/kaggle/working'  # 日志、worker 清单与隧道程序
DATASET_COPY_DIR = '/kaggle/working/see-through-dataset'  # 只读 Dataset 复制到此处再安装
ENV_DIR = '/tmp/see-through-venv'  # 独立推理环境，不继承 Kaggle 系统包；不占 working 保存空间
# ============ 鉴权与缓存 ============
HF_TOKEN = ''  # 模型公开，可留空；不会打印 token，也不保存 git credential
HF_HOME = '/kaggle/working/hf_cache'  # 模型缓存；首次下载约 8GB [INFERENCE]
# ============ 推理参数 ============
DTYPE = 'fp16'  # T4 不用 bf16；全黑/全透明时改 fp32，并把 RESOLUTION 降到 1024
CUDA_ALLOCATOR_CONF = 'expandable_segments:True'  # 避免 1280 fp32 VAE 解码时的显存碎片
RESOLUTION = 1280
RESOLUTION_DEPTH = 768  # fp32 仍 OOM 时可降到 512
STEPS = 30
SEED = 42
# ============ 批处理 ============
RUN_BATCH = False  # 默认只启动 Web；True 才执行 Cell 5–7，公网链接会等批处理全部完成后出现
INPUT_DIR = ''  # 空 = 仓库 test_image*.png；否则为图片目录绝对路径
SAVE_DIR = '/kaggle/working/layerdiff_output'
ARCHIVE_PATH = '/kaggle/working/psd_output.zip'
NUM_WORKERS = 2  # 每张卡一个进程；单卡自动收敛为 1
BATCH_GROUP_OFFLOAD = False  # 双 worker 不向共享 RAM 卸载权重；单 worker 可选 True
PEAK_VRAM_LIMIT_GB = 15  # Cell 6 的显存验收上限
ALPHA_MEAN_MIN = 0.5  # Cell 6 使用 0–255 alpha 的均值，不是归一化值
# ============ Web UI 与公网访问 ============
ENABLE_WEBUI = True  # False = 不启动 Web；只做批处理时还需 RUN_BATCH=True
TUNNEL = 'gradio'  # gradio | cloudflared | none（none 仍启动容器内服务）
WEB_RESOLUTION = 1024  # Web 滑块初始值；与批处理 RESOLUTION 独立
WEB_MAX_RESOLUTION = 2048  # UI 上限，可调；64 的倍数。单图仍只用一张 T4，不合并双卡显存
WEB_PORT = 7860
WEB_START_TIMEOUT = 120  # 秒；首次 FRP client 下载可能较慢
WEB_REQUEST_TIMEOUT = 5400  # 秒；每次 Web 推理的上限，映射 SEETHROUGH_REQUEST_TIMEOUT
WEB_DRAIN_GRACE = 60  # 秒；排空后保留文件端点，允许客户端接收最后一份输出
WEB_SESSION_SECONDS = 10800  # Cell 9 保活；到期停止接单并等待当前请求，0=跳过保活
POLL_INTERVAL = 5  # 检查服务/公网链接的间隔，秒
PROCESS_STOP_TIMEOUT = 15  # 秒；中断/重跑时等待 app 完成 worker（默认 5 秒）清理
CLOUDFLARED_URL = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
# ============ 独立运行环境的依赖版本 ============
BOOTSTRAP_PACKAGES = ('wrapt==1.17.3',)  # Kaggle 标准库 sitecustomize 的启动依赖
TORCH_PACKAGES = ('torch==2.8.0+cu128', 'torchvision==0.23.0+cu128')
TORCH_INDEX_URL = 'https://download.pytorch.org/whl/cu128'
# ============ 汇总打印（token 脱敏） ============
import json
print(json.dumps({k: ('***' if v else '') if k == 'HF_TOKEN' else v
                  for k, v in list(globals().items()) if k.isupper() and not k.startswith('_')},
                 indent=2, ensure_ascii=False, default=str))


In [ ]:
# Cell 3 · 获取仓库；HF_TOKEN 仅通过子进程环境变量使用，不写登录凭据
from pathlib import Path
import shutil, subprocess

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
repo_dir = Path(REPO_DIR)
if str(repo_dir.resolve()).startswith('/kaggle/input/'):
    # editable install 需要可写目录，不能直接安装只读 Dataset 中的 common。
    shutil.copytree(repo_dir, DATASET_COPY_DIR, dirs_exist_ok=True)
    repo_dir = Path(DATASET_COPY_DIR)
    print('使用 Dataset 可写副本:', repo_dir)
else:
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
    if (repo_dir / '.git').exists():
        subprocess.run(['git', '-C', str(repo_dir), 'remote', 'set-url', 'origin', REPO_URL], check=True)
        subprocess.run(['git', '-C', str(repo_dir), 'fetch', 'origin', REPO_BRANCH], check=True)
        remote_branch = subprocess.run(
            ['git', '-C', str(repo_dir), 'show-ref', '--verify', '--quiet',
             f'refs/remotes/origin/{REPO_BRANCH}']).returncode == 0
        if remote_branch:
            subprocess.run(['git', '-C', str(repo_dir), 'checkout', REPO_BRANCH], check=True)
            result = subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', REPO_BRANCH])
            if result.returncode:
                raise RuntimeError('无法 fast-forward：请保留/处理本地改动后重跑；未创建合并提交。')
        else:
            subprocess.run(['git', '-C', str(repo_dir), 'checkout', '--detach', 'FETCH_HEAD'], check=True)

inference_script = repo_dir / 'inference/scripts/inference_psd_quantized.py'
assert inference_script.is_file(), f'仓库路径不正确: {repo_dir}'
assert "add_argument('--dtype'" in inference_script.read_text(), (
    '此仓库尚无 --dtype：请在 Cell 2 改用包含 Kaggle 集成的 fork，'
    '或上传本仓库副本为 Kaggle Dataset 并修改 REPO_DIR。不要继续运行旧 main。')
for required in ('requirements-kaggle.txt', 'inference/scripts/app_gradio.py',
                 'inference/scripts/inference_worker.py', 'inference/scripts/inference_process.py'):
    assert (repo_dir / required).is_file(), f'缺少 {required}，请使用完整集成副本'
print('仓库自检通过:', repo_dir)


In [ ]:
# Cell 4 · 安装到干净 venv，不修改 Kaggle kernel 的预装依赖
import os, sys, subprocess, venv, json
venv.EnvBuilder(with_pip=False, system_site_packages=False, symlinks=True).create(ENV_DIR)
runtime_python = Path(ENV_DIR) / 'bin/python'
runtime_env = dict(os.environ, HF_HOME=HF_HOME, HF_TOKEN=HF_TOKEN,
                   PYTHONNOUSERSITE='1', MPLBACKEND='Agg')
runtime_env.pop('PYTHONPATH', None)
runtime_env.pop('PYTHONHOME', None)
# 先用主解释器定向安装启动依赖；不执行尚缺 wrapt 的 venv，也不写系统包。
runtime_site = Path(ENV_DIR) / 'lib' / f'python{sys.version_info.major}.{sys.version_info.minor}' / 'site-packages'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--target', str(runtime_site),
                *BOOTSTRAP_PACKAGES], env=runtime_env, check=True)
# pip --python 可管理没有安装 pip 的 venv，避免依赖系统 ensurepip。
installer = [sys.executable, '-m', 'pip', '--python', str(runtime_python)]
subprocess.run(installer + ['install', *TORCH_PACKAGES, '--index-url', TORCH_INDEX_URL],
               env=runtime_env, check=True)
subprocess.run(installer + ['install', '-q', '-r', 'requirements-kaggle.txt'],
               cwd=repo_dir, env=runtime_env, check=True)
subprocess.run(installer + ['check'], env=runtime_env, check=True)
runtime_info = json.loads(subprocess.check_output(
    [str(runtime_python), '-c',
     "import json, torch, matplotlib, pycocotools, gradio; "
     "print(json.dumps({'torch': torch.__version__, 'cuda': torch.version.cuda, "
     "'cuda_available': torch.cuda.is_available(), 'gradio': gradio.__version__}))"],
    env=runtime_env, text=True))
print('隔离环境:', runtime_python)
print(json.dumps(runtime_info, indent=2))
assert runtime_info['cuda_available'], '独立环境未检测到 CUDA，请检查 TORCH_PACKAGES / TORCH_INDEX_URL'
major, minor = map(int, runtime_info['torch'].split('+')[0].split('.')[:2])
allocator_key = 'PYTORCH_ALLOC_CONF' if (major, minor) >= (2, 11) else 'PYTORCH_CUDA_ALLOC_CONF'
subprocess.run([str(runtime_python), str(inference_script), '--help'],
               cwd=repo_dir, env=runtime_env, check=True)


In [ ]:
# Cell 5 · 双 worker 批处理（每卡一进程；结束后才可启动 Web UI）
if not RUN_BATCH:
    print('跳过 Cell 5 批处理：RUN_BATCH=False，继续启动 Web UI。')
else:
    import os, subprocess
    from pathlib import Path
    assert not (globals().get('web') and web.poll() is None), '先停止 Cell 8 的 Web UI，避免 GPU0 OOM'
    assert NUM_WORKERS >= 1, 'NUM_WORKERS 必须至少为 1'
    src_dir = Path(INPUT_DIR) if INPUT_DIR else repo_dir / 'common/assets'
    imgs = sorted(p.resolve() for p in src_dir.iterdir()
                  if p.is_file() and p.suffix.lower() in ('.png', '.jpg', '.jpeg', '.webp'))
    if not INPUT_DIR:
        imgs = [p for p in imgs if p.name.startswith('test_image')]
    assert imgs, f'没有找到输入图片: {src_dir}'
    assert len({p.stem for p in imgs}) == len(imgs), '输入文件名去掉扩展名后必须唯一，否则输出会覆盖'
    n = min(NUM_WORKERS, torch.cuda.device_count())
    assert n > 0, '没有可用 GPU'
    buckets = [imgs[i::n] for i in range(n)]
    Path(SAVE_DIR).mkdir(parents=True, exist_ok=True)
    # 用 argv 而不是 shell 拼接，支持带空格的路径，并且每张图片失败都会传回非零退出码。
    worker_code = (
        "import subprocess, sys\n"
        "from pathlib import Path\n"
        "for image in Path(sys.argv[1]).read_text().splitlines():\n"
        "    subprocess.run(sys.argv[2:] + ['--srcp', image], check=True)\n"
    ).replace('\\n', '\n')
    procs, worker_logs = [], []
    for gpu, bucket in enumerate(buckets):
        if not bucket:
            continue
        listf = Path(WORK_DIR) / f'list_gpu{gpu}.txt'
        listf.write_text('\n'.join(map(str, bucket)) + '\n')
        logp = Path(WORK_DIR) / f'worker{gpu}.log'
        cmd = [str(runtime_python), '-u', '-c', worker_code, str(listf), str(runtime_python), '-u',
               str(inference_script), '--quant_mode', 'none', '--dtype', DTYPE,
               '--resolution', str(RESOLUTION), '--resolution_depth', str(RESOLUTION_DEPTH),
               '--num_inference_steps', str(STEPS), '--seed', str(SEED),
               '--save_dir', SAVE_DIR, '--save_to_psd']
        cmd += ['--group_offload'] if BATCH_GROUP_OFFLOAD else ['--no_group_offload']
        env = dict(runtime_env, CUDA_VISIBLE_DEVICES=str(gpu), HF_HOME=HF_HOME, HF_TOKEN=HF_TOKEN)
        env[allocator_key] = CUDA_ALLOCATOR_CONF
        with logp.open('w') as log:
            procs.append(subprocess.Popen(cmd, cwd=repo_dir, env=env, stdout=log, stderr=subprocess.STDOUT))
        worker_logs.append(logp)
        print(f'GPU{gpu}: {len(bucket)} 张图片 → {logp}')
    codes = [p.wait() for p in procs]
    for logp in worker_logs:
        print(f'\n--- {logp.name} ---\n' + '\n'.join(logp.read_text().splitlines()[-20:]))
    assert all(code == 0 for code in codes), f'worker 失败: {codes}；查看以上日志'


In [ ]:
# Cell 6 · 端到端结果校验；不以“进程没报错”代替图片/PSD 验收
if not RUN_BATCH:
    print('跳过 Cell 6 批处理结果校验：RUN_BATCH=False。')
else:
    import json
    # 第三方 PSD/图片校验也在隔离环境运行，不向 notebook kernel 安装 psd-tools。
    validation_code = r'''
import json, math, sys
from pathlib import Path
import numpy as np
from PIL import Image
from psd_tools import PSDImage
config = json.load(sys.stdin)
imgs = [Path(p) for p in config['images']]
SAVE_DIR, DTYPE = config['save_dir'], config['dtype']
PEAK_VRAM_LIMIT_GB, ALPHA_MEAN_MIN = config['vram_limit'], config['alpha_min']
fallback = "请改 Cell 2: DTYPE='fp32' + RESOLUTION=1024、NUM_WORKERS=1；Web 同时降低 WEB_RESOLUTION；仍 OOM 则 RESOLUTION_DEPTH=512。"
for image_path in imgs:
    saved = Path(SAVE_DIR) / image_path.stem
    psd_path = saved.with_suffix(saved.suffix + '.psd')
    assert psd_path.is_file(), f'缺少 {psd_path}。{fallback}'
    psd = PSDImage.open(psd_path)
    assert len(psd) > 0, f'PSD 没有分层: {psd_path}。{fallback}'
    stats = json.loads((saved / 'stats.json').read_text())
    assert stats['dtype'] == DTYPE and stats['quant_mode'] == 'none', 'stats 与本次配置不一致，请重跑 Cell 5'
    assert math.isfinite(stats['peak_vram_gb']) and 0 < stats['peak_vram_gb'] <= PEAK_VRAM_LIMIT_GB, (
        f"peak VRAM={stats['peak_vram_gb']:.2f}GB 超出验收范围。{fallback}")
    assert math.isfinite(stats['total_time_s']) and stats['total_time_s'] > 0
    layers = [p for p in sorted(saved.glob('*.png'))
              if p.stem not in {'src_img', 'src_head', 'reconstruction'} and not p.stem.endswith('_depth')]
    assert layers, f'未生成分解 PNG。{fallback}'
    samples = []
    for layer_path in layers:
        with Image.open(layer_path) as layer:
            rgba = np.array(layer.convert('RGBA'))
        samples.append((float(rgba[..., 3].mean()), layer_path, rgba))
    alpha_mean, sample_path, rgba = max(samples, key=lambda item: item[0])
    assert alpha_mean > ALPHA_MEAN_MIN, f'输出全透明: {sample_path}。{fallback}'
    visible = rgba[..., 3] > 0
    assert np.any(rgba[..., :3][visible] > 0), f'输出全黑: {sample_path}。{fallback}'
    print(f"{image_path.name}: {len(psd)} layers, peak VRAM {stats['peak_vram_gb']:.2f}GB, "
          f"total {stats['total_time_s']:.1f}s, sample {sample_path.name}, alpha mean {alpha_mean:.2f}")
print('全部结果校验通过')
'''
    subprocess.run([str(runtime_python), '-c', validation_code], env=runtime_env, text=True, check=True,
                   input=json.dumps({'images': list(map(str, imgs)), 'save_dir': SAVE_DIR, 'dtype': DTYPE,
                                     'vram_limit': PEAK_VRAM_LIMIT_GB, 'alpha_min': ALPHA_MEAN_MIN}))


In [ ]:
# Cell 7 · 打包 PSD（包含深度 PSD）；使用 SAVE_DIR/ARCHIVE_PATH，不写死输出目录
if not RUN_BATCH:
    print('跳过 Cell 7 批处理打包：RUN_BATCH=False。')
else:
    from zipfile import ZipFile, ZIP_DEFLATED
    from IPython.display import FileLink, display
    psds = sorted(Path(SAVE_DIR).rglob('*.psd'))
    assert psds, '没有 PSD 可打包，请先完成 Cell 5–6'
    Path(ARCHIVE_PATH).parent.mkdir(parents=True, exist_ok=True)
    with ZipFile(ARCHIVE_PATH, 'w', compression=ZIP_DEFLATED) as archive:
        for psd_path in psds:
            archive.write(psd_path, arcname=str(psd_path.relative_to(SAVE_DIR)))
    with ZipFile(ARCHIVE_PATH) as archive:
        assert all(image_path.stem + '.psd' in archive.namelist() for image_path in imgs)
    print('下载归档:', ARCHIVE_PATH)
    display(FileLink(ARCHIVE_PATH))


In [ ]:
# Cell 8 · Web UI；每次请求启动独立推理进程，重跑先终止旧服务/隧道
import os, subprocess, time, re, signal
from urllib.request import urlopen
from urllib.error import URLError
assert TUNNEL in ('gradio', 'cloudflared', 'none'), 'TUNNEL 只能是 gradio / cloudflared / none'
assert not any(p.poll() is None for p in globals().get('procs', [])), '先等待批处理 worker 结束'
for old in (globals().get('web'), globals().get('cf')):
    if old is not None and old.poll() is None:
        old.terminate()
        try:
            old.wait(timeout=PROCESS_STOP_TIMEOUT)
        except subprocess.TimeoutExpired:
            old.kill()
            old.wait()
if not ENABLE_WEBUI:
    print('Web UI disabled: ENABLE_WEBUI=False')
else:
    env = dict(runtime_env, CUDA_VISIBLE_DEVICES='0', HF_HOME=HF_HOME, HF_TOKEN=HF_TOKEN,
               PYTHONUNBUFFERED='1', PORT=str(WEB_PORT), SEETHROUGH_DTYPE=DTYPE,
               SEETHROUGH_SHARE='1' if TUNNEL == 'gradio' else '0',
               SEETHROUGH_REQUEST_TIMEOUT=str(WEB_REQUEST_TIMEOUT),
               SEETHROUGH_DRAIN_GRACE=str(WEB_DRAIN_GRACE),
               SEETHROUGH_MAX_RESOLUTION=str(WEB_MAX_RESOLUTION),
               SEETHROUGH_RESOLUTION=str(WEB_RESOLUTION), SEETHROUGH_STEPS=str(STEPS),
               SEETHROUGH_SEED=str(SEED), SEETHROUGH_DEPTH_RESOLUTION=str(RESOLUTION_DEPTH))
    env[allocator_key] = CUDA_ALLOCATOR_CONF
    web_log = Path(WORK_DIR) / 'webui.log'
    with web_log.open('w') as log:
        web = subprocess.Popen([str(runtime_python), '-u', 'inference/scripts/app_gradio.py'],
                               cwd=repo_dir, env=env, stdout=log, stderr=subprocess.STDOUT)
    local_url = f'http://127.0.0.1:{WEB_PORT}'
    deadline = time.monotonic() + WEB_START_TIMEOUT
    ready = False
    public_url = None
    while time.monotonic() < deadline:
        if web.poll() is not None:
            raise RuntimeError('Web UI 启动失败:\n' + web_log.read_text())
        text = web_log.read_text()
        match = re.search(r'https://[a-z0-9-]+\.gradio\.live', text)
        if match:
            public_url = match.group(0)
        try:
            with urlopen(local_url, timeout=POLL_INTERVAL) as response:
                ready = response.status == 200
        except (URLError, TimeoutError):
            pass
        if ready and (TUNNEL != 'gradio' or public_url):
            break
        time.sleep(POLL_INTERVAL)
    assert ready, f'Web UI 未就绪，查看 {web_log}'
    if TUNNEL == 'gradio':
        if public_url:
            print('公网地址:', public_url)
        else:
            print(f'未获取到 gradio.live 链接：查 https://status.gradio.app 或 {web_log}；改 TUNNEL="cloudflared" 后重跑本格及 Cell 9')
    elif TUNNEL == 'cloudflared':
        print('Web UI 已就绪，请运行 Cell 9 创建备用隧道')
    else:
        print('仅容器内地址:', local_url, '（Kaggle 无入站端口，外部浏览器不能直连）')


In [ ]:
# Cell 9 · 备用 cloudflared + CLI 公网服务保活；需先跑 Cell 8
from urllib.request import urlretrieve
if ENABLE_WEBUI and TUNNEL == 'cloudflared':
    assert globals().get('web') and web.poll() is None, '请先运行 Cell 8 启动 Web UI'
    if globals().get('cf') and cf.poll() is None:
        cf.terminate()
        try:
            cf.wait(timeout=PROCESS_STOP_TIMEOUT)
        except subprocess.TimeoutExpired:
            cf.kill()
            cf.wait()
    binary = Path(WORK_DIR) / 'cloudflared'
    urlretrieve(CLOUDFLARED_URL, binary)
    binary.chmod(binary.stat().st_mode | 0o111)
    cf_log = Path(WORK_DIR) / 'cf.log'
    with cf_log.open('w') as log:
        cf = subprocess.Popen([str(binary), 'tunnel', '--url', f'http://127.0.0.1:{WEB_PORT}'],
                              stdout=log, stderr=subprocess.STDOUT)
    deadline = time.monotonic() + WEB_START_TIMEOUT
    while time.monotonic() < deadline:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', cf_log.read_text())
        if match:
            print('公网地址:', match.group(0))
            break
        if cf.poll() is not None:
            raise RuntimeError('cloudflared 退出:\n' + cf_log.read_text())
        time.sleep(POLL_INTERVAL)
    else:
        print('未获取到备用隧道链接，查看', cf_log, '；两种隧道均失败可设 TUNNEL="none"，批处理不受影响')
else:
    print('无需 cloudflared；当前 TUNNEL =', TUNNEL)

# CLI / Save & Run All 在 notebook 结束后会释放 session；公网服务需显式保活。
# 交互运行只跑 Cell 8 即可；本格到期先 drain，中断则终止当前请求并关闭服务。
if ENABLE_WEBUI and TUNNEL != 'none' and WEB_SESSION_SECONDS > 0:
    from datetime import datetime, timedelta, timezone
    assert globals().get('web') and web.poll() is None, '请先运行 Cell 8 启动 Web UI'
    expires = datetime.now(timezone.utc) + timedelta(seconds=WEB_SESSION_SECONDS)
    print('公网服务停止接单时间 (UTC):', expires.isoformat(), flush=True)
    deadline = time.monotonic() + WEB_SESSION_SECONDS
    try:
        with web_log.open() as live_log:
            while time.monotonic() < deadline:
                chunk = live_log.read()
                if chunk:
                    print(chunk, end='', flush=True)
                if web.poll() is not None:
                    raise RuntimeError('Web UI 提前退出，请查看 webui.log')
                time.sleep(POLL_INTERVAL)
            # 正常到期保留隧道：app 拒绝新请求，当前请求沿用已开始的超时。
            web.send_signal(signal.SIGUSR1)
            print('服务停止接单，等待当前请求完成…', flush=True)
            drain_deadline = time.monotonic() + WEB_REQUEST_TIMEOUT + WEB_DRAIN_GRACE + PROCESS_STOP_TIMEOUT
            while web.poll() is None and time.monotonic() < drain_deadline:
                chunk = live_log.read()
                if chunk:
                    print(chunk, end='', flush=True)
                time.sleep(POLL_INTERVAL)
            chunk = live_log.read()
            if chunk:
                print(chunk, end='', flush=True)
            if web.poll() is None:
                print('排空等待超时，终止服务及当前 worker。', flush=True)
    finally:
        # 中断/异常/排空超时：SIGTERM → 有界等待 → SIGKILL。
        # app 负责 worker teardown；强杀 app 时 worker 的 stdin EOF 守卫兜底。
        for child in (web, globals().get('cf')):
            if child is not None and child.poll() is None:
                child.terminate()
                try:
                    child.wait(timeout=PROCESS_STOP_TIMEOUT)
                except subprocess.TimeoutExpired:
                    child.kill()
                    child.wait()


## Cell 10 · Web UI 注意事项

- 公网服务生命周期 = Kaggle session 生命周期：单 session **12h** 硬上限；交互 session 约 **20 分钟空闲**可能断开。持续批任务可用 **Save & Run All**；交互 UI 需保持 notebook 活动。外部访问隧道不等于保持 Kaggle session 活动。
- CLI / Save & Run All 会执行 Cell 9 保活公网服务，默认 `WEB_SESSION_SECONDS=10800`（3h）。正常到期发送 SIGUSR1 停止接单，保留隧道等待当前请求（仍受其原有 `WEB_REQUEST_TIMEOUT=5400` 秒上限约束），完成输出交接后再保留下载端点 `WEB_DRAIN_GRACE=60` 秒；最多再等待请求上限加下载宽限及停止余量。中断或重跑则发送 SIGTERM，等待 15 秒后必要时强杀。0 表示跳过保活。交互 Gradio 只需 Cell 8，cloudflared 还需 Cell 9。
- 当前为无鉴权公网服务，上传图片及输出会暂存在 Kaggle。请勿上传敏感或未经授权的内容。`queue(max_size=16)` 仅限制排队长度，**不是鉴权或防滥用机制**。输出缓存在 Gradio 中按小时清理，请及时下载。
- Gradio share 状态：[status.gradio.app](https://status.gradio.app/)。不可用时在 Cell 2 设 `TUNNEL='cloudflared'`，依次重跑 Cell 8–9；两者均失败设 `TUNNEL='none'` 只保留容器内服务，或 `ENABLE_WEBUI=False` 关闭服务。
- Web UI **只用 GPU0**。不要与 Cell 5 批处理同时跑；第二张卡仅在双 worker 批处理时使用。父进程只保留队列和输出缓存，每次请求在新解释器加载模型，完成、OOM 或超时后退出；不跨请求保留管线。
- 高分辨率：`WEB_MAX_RESOLUTION` 默认 2048，`WEB_RESOLUTION` 默认 1024，按 64 像素步长调整；选择 2K 时保留 30 steps。两张卡不合并显存。当前逐请求隔离架构的 19 个预定 HTTP 请求已完成有限序列验收，两次 2048 / 30 steps 均成功（约 65 分钟／次），2560 CUDA OOM 后新 worker 的 1536 也成功；不降低单次峰值、不保证任意输入或长期可用，也不修复 PSD alpha／预览毛边。完整方法与边界见 `kaggle/VALIDATION.md`。
- 从**非 Kaggle 的浏览器**打开公网链接，上传 `common/assets/test_image.png`，确认返回可下载且有分层的 PSD、gallery 非空。这一步才是公网推理验收；启动成功不代表 GPU 推理已验证。


## Cell 11 · 配额、预期与故障定位

- 常见免费 GPU 配额 **30h/周**、单 session **12h**；以账号当前界面和 [Kaggle notebook 文档](https://www.kaggle.com/docs/notebooks) 为准。T4×2 通常配 4 CPU / 29GB RAM；`/kaggle/working` 自动保存限额约 20GB，模型缓存与 PSD 会占空间。
- 模型公开，`HF_TOKEN=''` 可用。首次从 Hugging Face 下载约 **8GB [INFERENCE]** 至 `HF_HOME`；首次启动 Gradio share 也需要联网下载 FRP client。
- T4 无 bf16 硬件支持：本集成用 **quant_mode=none + fp16 compute + fp32 VAE**，不安装 bitsandbytes。现有 NF4 checkpoint 的 compute dtype 为 bf16，不适合直接用于 T4。
- 2026-10-26 历史实测（基础环境 torch 2.11.0+cu128，并非当前隔离环境）：T4×2、1280、30 steps、fp16 compute / fp32 VAE，两图各卡并行均成功，单图总耗时约 **16.9 / 18.6 分钟**，各 **17 层 PSD**、peak VRAM **13.01 GiB**。使用 `BATCH_GROUP_OFFLOAD=False` 与 expandable segments；同次共享内存峰值约 **29.04 GiB**，OOM 计数为 0。这是两张样例，不保证其他运行时或输入同样耗时/峰值。
- 2026-10-27 **旧长驻服务历史实测**：torch **2.8.0+cu128**，目标 `pip check` 通过、基础包清单不变；公网 Web 用 **1536 / 10 steps / seed 42**、当时的 `KEEP_RESIDENT=False` 完成请求，下载并渲染了 **1536×1536、17 层 PSD**，gallery **24 项**，首次请求 **937.9 秒**。后续旧服务的连续请求与 OOM 后恢复失败；不要与当前逐请求隔离架构的有限序列验收混用。双 worker 高分辨率仍未验证。详见 `kaggle/VALIDATION.md`。
- `RUN_BATCH=True` 时，期望 Cell 5 的每个非空 worker 日志都有 `LayerDiff3D done`、`Marigold done`、`PSD assembly done`；Cell 6 全过，Cell 7 产生含每图 PSD 的归档。默认 Web 模式会明确跳过这三个单元。
- 输出全黑/透明：只改 Cell 2 为 `DTYPE='fp32'`、`RESOLUTION=1024`，Web 也降 `WEB_RESOLUTION`；仍 OOM 时降 `RESOLUTION_DEPTH=512`，然后重跑相关格子。Cell 6 能捕获明显的空输出，不能代替人工质量检查。
- 双 worker 开 group offload 会把权重卸载到共享主机内存，实测触及 30 GiB 上限并触发 OS OOM kill；批处理默认关闭。关闭后仍需保留 `CUDA_ALLOCATOR_CONF` 防止 1280 解码显存碎片。fp32 回退时建议同时设 `NUM_WORKERS=1`；更大的显存/内存需求未验证。
- Cell 4 只向 `ENV_DIR` 独立 venv 安装固定 CUDA PyTorch 和精简依赖；期望目标环境 `pip check` 输出 `No broken requirements found.`。修改版本后重跑 Cell 4，再重启相关子进程，不必重启基础 kernel。不要把 requirements 装入基础 kernel，也不要为了无关预装包的冲突升级/降级系统依赖。
- CLI 的 notebook ID / Dataset 挂载属于平台提交元数据，修改 `kernel-metadata.json`；推理与服务参数全部只改 Cell 2。Dataset 是只读源，Cell 3 会复制到 `DATASET_COPY_DIR` 后使用。
- 上游维护与同步清单见仓库 [`kaggle/README.md`](https://github.com/shitagaki-lab/see-through/tree/main/kaggle)。本机无 NVIDIA GPU；GPU PSD 推理与两种公网隧道需在 Kaggle 按上述单元格验收。
